# Hindi hostility — extra runs for the IEEE version (v3)
**What this does:** trains mBERT and MuRIL with two more random seeds (13 and 77), on raw and cleaned text, for the coarse and fine tasks; saves every test prediction; and runs the URL test on each raw coarse model.

**How to run:** Runtime → Change runtime type → **T4 GPU** → Save. Then **Runtime → Run all**. Allow Google Drive when asked. No Hugging Face token is needed.

Takes about 3 hours. Everything is saved in Drive (MyDrive/hostility) after each model, so after a disconnect just open the notebook again and press Run all: finished runs are skipped.

When it ends, send these files from MyDrive/hostility: `v3_results.csv`, `v3_url_test.csv` and the folder `v3_preds`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/hostility/v3_preds
%cd /content/drive/MyDrive/hostility

In [ ]:
!pip -q install transformers accelerate scikit-learn pandas
![ -d Hindi-Hostility-Detection-CONSTRAINT-2021 ] || git clone -q --depth 1 https://github.com/mohit19014/Hindi-Hostility-Detection-CONSTRAINT-2021.git

In [ ]:
import os, re, time, json, random, string, gc
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from sklearn.metrics import f1_score, accuracy_score
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
D = 'Hindi-Hostility-Detection-CONSTRAINT-2021/Dataset'
FINE = ['fake','hate','offensive','defamation']
tr, va, te = (pd.read_csv(f'{D}/{s}.csv') for s in ['train','valid','test'])
def labels(df):
    sets = df['Labels Set'].str.split(',').apply(lambda l: {x.strip() for x in l})
    return (~sets.apply(lambda s: 'non-hostile' in s)).astype(int).values, np.stack([sets.apply(lambda s, l=l: int(l in s)).values for l in FINE], 1)
(ytr_c, ytr_f), (yva_c, yva_f), (yte_c, yte_f) = labels(tr), labels(va), labels(te)
URL = re.compile(r'https?://\S+|www\.\S+'); MEN = re.compile(r'@\w+'); HASH = re.compile(r'#\S+')
clean = lambda s: HASH.sub(' ', MEN.sub(' ', URL.sub(' ', s)))
RAW = {k: d['Post'].fillna('').astype(str).tolist() for k, d in [('tr',tr),('va',va),('te',te)]}
TEXT = {'raw': RAW, 'clean': {k: [clean(x) for x in v] for k, v in RAW.items()}}
MODELS = {'mBERT':'bert-base-multilingual-cased', 'MuRIL':'google/muril-base-cased'}
SEEDS = [13, 77]
EPOCHS, LR, BS, MAXLEN, dev = 4, 2e-5, 16, 128, 'cuda'
GRID = np.arange(0.05, 0.96, 0.05)
print('data ok', len(tr), len(va), len(te))

In [ ]:
def enc(tok, texts):
    e = tok(texts, truncation=True, max_length=MAXLEN, padding='max_length', return_tensors='pt')
    return e['input_ids'], e['attention_mask']

def train(path, T, y_tr, y_va, multilabel, seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    tok = AutoTokenizer.from_pretrained(path)
    m = AutoModelForSequenceClassification.from_pretrained(path, num_labels=4 if multilabel else 2).to(dev)
    ids, am = enc(tok, T['tr']); yt = torch.tensor(y_tr, dtype=torch.float if multilabel else torch.long)
    g = torch.Generator(); g.manual_seed(seed)
    dl = DataLoader(TensorDataset(ids, am, yt), batch_size=BS, shuffle=True, generator=g)
    opt = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=0.01)
    sch = get_linear_schedule_with_warmup(opt, int(0.1*len(dl)*EPOCHS), len(dl)*EPOCHS)
    pw = torch.tensor((len(y_tr) - y_tr.sum(0)) / y_tr.sum(0), dtype=torch.float, device=dev) if multilabel else None
    def scores(texts):
        i, a = enc(tok, texts); m.eval(); out = []
        with torch.no_grad():
            for b in range(0, len(i), 64):
                lg = m(input_ids=i[b:b+64].to(dev), attention_mask=a[b:b+64].to(dev)).logits
                out.append(torch.sigmoid(lg).cpu() if multilabel else torch.softmax(lg, -1)[:, 1].cpu())
        return torch.cat(out).numpy()
    best, state, th = -1, None, None
    t0 = time.perf_counter()
    for ep in range(EPOCHS):
        m.train()
        for i, a, y in dl:
            lg = m(input_ids=i.to(dev), attention_mask=a.to(dev)).logits
            loss = F.binary_cross_entropy_with_logits(lg, y.to(dev), pos_weight=pw) if multilabel else F.cross_entropy(lg, y.to(dev))
            loss.backward(); torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step(); sch.step(); opt.zero_grad()
        pv = scores(T['va'])
        if multilabel:
            t = [max(GRID, key=lambda x: f1_score(y_va[:,j], (pv[:,j] > x).astype(int))) for j in range(4)]
            s = np.mean([f1_score(y_va[:,j], (pv[:,j] > t[j]).astype(int)) for j in range(4)])
        else:
            t, s = None, f1_score(y_va, (pv > 0.5).astype(int), average='macro')
        print(f'  epoch {ep+1}: validation {s:.4f}')
        if s > best: best, state, th = s, {k: v.detach().clone() for k, v in m.state_dict().items()}, t
    m.load_state_dict(state)
    return m, tok, scores, th, time.perf_counter() - t0

def load(f): return pd.read_csv(f) if os.path.exists(f) else pd.DataFrame()
def done(df, **kw):
    if df.empty: return False
    mask = np.ones(len(df), bool)
    for k, v in kw.items(): mask &= (df[k] == v).values
    return mask.any()

In [ ]:
rng = np.random.default_rng(0); AL = list(string.ascii_letters + string.digits)
fake_url = lambda: 'https://t.co/' + ''.join(rng.choice(AL, 10))
fake_str = lambda: ''.join(rng.choice(list(string.ascii_lowercase), 23))

for seed in SEEDS:
  for variant in ['raw', 'clean']:
    for name, path in MODELS.items():
      for task in ['coarse', 'fine']:
        R = load('v3_results.csv')
        if done(R, model=name, text=variant, task=task, seed=seed):
            print('skip', name, variant, task, seed); continue
        print('RUN', name, variant, task, seed)
        T = TEXT[variant]; ml = task == 'fine'
        m, tok, scores, th, tt = train(path, T, ytr_f if ml else ytr_c, yva_f if ml else yva_c, ml, seed)
        p = scores(T['te'])
        if ml:
            pf = np.stack([(p[:,j] > th[j]).astype(int) for j in range(4)], 1)
            per = [f1_score(yte_f[:,j], pf[:,j]) for j in range(4)]; sup = yte_f.sum(0)
            r = dict(model=name, text=variant, task=task, seed=seed, train_s=tt, fine_weightedF1=float(np.dot(per, sup)/sup.sum()),
                     **{f'F1_{l}': v for l, v in zip(FINE, per)}, thresholds=str([round(float(x),2) for x in th]))
            np.save(f'v3_preds/{name}_{variant}_{task}_{seed}.npy', pf)
        else:
            pc = (p > 0.5).astype(int)
            r = dict(model=name, text=variant, task=task, seed=seed, train_s=tt, coarse_acc=accuracy_score(yte_c, pc),
                     coarse_macroF1=f1_score(yte_c, pc, average='macro'), coarse_weightedF1=f1_score(yte_c, pc, average='weighted'))
            np.save(f'v3_preds/{name}_{variant}_{task}_{seed}.npy', pc)
            if variant == 'raw':   # URL test on the raw coarse model
                te_txt = RAW['te']; hu = np.array([bool(URL.search(t)) for t in te_txt])
                A = np.where((yte_c == 1) & (pc == 1) & ~hu)[0]; B = np.where((yte_c == 0) & (pc == 0) & hu)[0]
                pa = scores([te_txt[i] + ' ' + fake_url() for i in A]) > 0.5
                pcn = scores([te_txt[i] + ' ' + fake_str() for i in A]) > 0.5
                pb = scores([URL.sub(' ', te_txt[i]) for i in B]) > 0.5
                u = dict(model=name, seed=seed, n_add=len(A), add_url_flip_pct=100*float((~pa).mean()),
                         add_control_flip_pct=100*float((~pcn).mean()), n_remove=len(B), remove_url_flip_pct=100*float(pb.mean()))
                pd.concat([load('v3_url_test.csv'), pd.DataFrame([u])]).to_csv('v3_url_test.csv', index=False); print(u)
        pd.concat([R, pd.DataFrame([r])]).to_csv('v3_results.csv', index=False); print(r)
        del m; gc.collect(); torch.cuda.empty_cache()
print('ALL DONE'); print(load('v3_results.csv')); print(load('v3_url_test.csv'))